In [ ]:
import dai
import pandas as pd

bsd = '2024-01-01'
sd = '2025-01-01'
ed = '2025-12-31'

sql = """
SELECT
    date, instrument,
    close/m_lag(close, 1) as factor,
    close/m_lag(close, 2) as factor2,
    close/m_lag(close, 3) as factor3,
    close/m_lag(close, 4) as factor4,
    close/m_lag(close, 5) as factor5,
    close/m_lag(close, 6) as factor6,
    close/m_lag(close, 7) as factor7,
    close/m_lag(close, 8) as factor8,
    close/m_lag(close, 9) as factor9,
    close/m_lag(close, 10) as factor10,
FROM cn_stock_bar1d
"""

df = dai.query(sql, filters={'date': [bsd, ed]}).df()
df = df[(df['date']>=sd) & (df['date']<=ed)]

index_df = dai.query(
    "SELECT date, member_code as instrument FROM cn_stock_index_component WHERE instrument = '000852.SH'",
    filters={"date": [sd, ed]},
).df()

df = pd.merge(df, index_df, how='right', on=['date', 'instrument'])

In [ ]:
import os
import sys
path = '/home/aiuser/work/workspace/BigAlpha/eval/76ad3f56-ec2b-431a-890e-139a7f4bbcba/src/bigalpha_factorminer'
if path not in sys.path:
    sys.path.append(path)

In [ ]:
from dataprocess.datachecker import DataCheck
from dataprocess.dataprocess import DataProcess
DataCheck(sd, ed).validate(df)
pdf = DataProcess(sd, ed).validate(df)

In [ ]:
from factoranalyze import FactorAnalyze
fa_res = FactorAnalyze(sd, ed).score(df[['date', 'instrument', 'factor']])

In [ ]:
from regmodel import ElasticNetRegress

reg = ElasticNetRegress(sd, ed)
reg_res = reg.score(pdf)